🐍 PyTorch Bootcamp — LESSON 8
Transfer Learning & Pretrained CNNs with PyTorch

### Import Libraries

In [1]:
# load torchvision library
from torchvision import models

In [2]:
# load a pre-trained ResNet18 
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights) 

# This tells PyTorch:
# Load ResNet18 with its pretrained weights

In [3]:
# inspect the model architecture
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
# inspect model.fc
print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


## Replace the Classifier

#### For CIFAR-10:

In [5]:
import torch.nn as nn

model.fc = nn.Linear(
    model.fc.in_features,  # number of input features
    10  # number of output classes
)

In [6]:
print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


In [7]:
# freeze its parameters
for parameter in model.parameters():
    parameter.requires_grad = False

In [8]:
# unfreeze the parameters
for parameter in model.fc.parameters():
    parameter.requires_grad = True

In [9]:
# image preprocessing
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

In [10]:
# print the preprocess transforms
print(preprocess) # This gives the preprocessing pipeline associated with the pretrained weights.

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [11]:
# CIFAR-10 images are:

# 32 × 32

# while pretrained ImageNet models are generally used with larger images, commonly around:

# 224 × 224

# Therefore, when using pretrained ResNet with CIFAR-10, we'll resize the images.

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
# why normalize
# The pretrained ResNet expects inputs normalized approximately using:

# Mean:
# [0.485, 0.456, 0.406]

# Std:
# [0.229, 0.224, 0.225]

# so we use: # This is important when using ImageNet pretrained weights.
transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

In [13]:
# training transform
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# testing transform
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## Load the CIFAR-10 Dataset

In [14]:
# no 26.
from torchvision import datasets

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

Files already downloaded and verified
Files already downloaded and verified


### Load DataLoaders

In [15]:
# create DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

### Move to CPU

In [16]:
import torch

device = torch.device("cpu")
model = model.to(device)

### Check Trainable Parameters

In [17]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Trainable Parameters: ",
    trainable_parameters
)

print(
    "Total Parameters: ",
    total_parameters
)

Trainable Parameters:  5130
Total Parameters:  11181642


#### Loss Function

In [18]:
# loss function
loss_fn = nn.CrossEntropyLoss()

#### Optimizer

In [19]:
# optitmizer - training a new classifier
optimizer = torch.optim.Adam(
    model.fc.parameters(), # optimize the new classifier
    lr=0.001
)

### Training Function

In [20]:
def train_step(
    model,
    dataloader,
    loss_fn,
    optimizer,
    device
):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(X)

        loss = loss_fn(
            logits,
            y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == y
        ).sum().item()

        total += y.size(0)

    average_loss = (
        total_loss /
        len(dataloader)
    )

    accuracy = correct / total

    return average_loss, accuracy

### Validation Function

In [21]:
def validation_step(
    model,
    dataloader,
    loss_fn,
    device
):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, y in dataloader:

            X = X.to(device)
            y = y.to(device)

            logits = model(X)

            loss = loss_fn(
                logits,
                y
            )

            total_loss += loss.item()

            predictions = logits.argmax(
                dim=1
            )

            correct += (
                predictions == y
            ).sum().item()

            total += y.size(0)

    average_loss = (
        total_loss /
        len(dataloader)
    )

    accuracy = correct / total

    return average_loss, accuracy

### Model Training

In [22]:
# now we can train the model using the train_step and validation_step functions. We will also keep track of the training and validation loss and accuracy for each epoch.
# no of epochs
epochs = 1

In [23]:
for epoch in range(epochs):

    train_loss, train_acc = train_step(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    test_loss, test_acc = validation_step(
        model,
        test_loader,
        loss_fn,
        device
    )

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )

Epoch 1/1 | Train Loss: 0.8025 | Train Acc: 0.7316 | Test Loss: 0.6025 | Test Acc: 0.7967


## Save the Model

In [24]:
torch.save(
    model.state_dict(),
    "ResNet18_CIFAR10.pth"
)

## Loading the Model

In [25]:
# reconstruct the same architecture
weights = ResNet18_Weights.DEFAULT

model = resnet18(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    10
)

In [26]:
model.load_state_dict(
    torch.load(
        "ResNet18_CIFAR10.pth",
        map_location=device,
        weights_only=True
    )
)

<All keys matched successfully>

In [27]:
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Make Predictions

In [28]:
image, label = test_dataset[0]

image_batch = image.unsqueeze(0)

model.eval()

with torch.no_grad():

    logits = model(
        image_batch.to(device)
    )

prediction = logits.argmax(
    dim=1
).item()

    



In [29]:
print(
    "Actual: ",
    test_dataset.classes[label]
)

print(
    "Predicted: ",
    test_dataset.classes[prediction]
)

Actual:  cat
Predicted:  cat


In [30]:
# 42 prediciton probability
probabilities = torch.softmax(
    logits,
    dim=1
)

confidence = probabilities[
    0,
    prediction
].item()

print(
    f"Confidence: {confidence:.4f}"
)

Confidence: 0.6189


ResNet18 has Layer 1 - Layer 4

##### Unfreeze the final residual netork

In [31]:
for parameter in model.parameters():
    parameter.requires_grad = False

##### Unfreeze

In [32]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

In [33]:
for parameter in model.fc.parameters():
    parameter.requires_grad = True